# Viettel RAG — Colab Notebook

Chạy pipeline RAG: retrieval + LLM Qwen3-4B trên 100 example questions.

In [ ]:
# ── Cell 1: Clone repo ──────────────────────────────────────────────────────
!git clone https://github.com/thekhiem14/Viettel_RAG.git
%cd /content/Viettel_RAG
!git checkout Khiem

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
# Core
!pip install -q FlagEmbedding faiss-gpu pyvi rank_bm25 rapidfuzz
# LLM quantization
!pip install -q transformers accelerate bitsandbytes>=0.41.0
# Utils
!pip install -q python-dotenv pandas

In [ ]:
# ── Cell 3: Verify artifacts (đã commit sẵn, không cần rebuild) ─────────────
import os
from pathlib import Path

artifacts = [
    'artifacts/docs/chunks.jsonl',
    'artifacts/docs/faiss.index',
    'artifacts/docs/bm25.pkl',
    'artifacts/api/faiss.index',
    'artifacts/api/bm25.pkl',
    'artifacts/api/fuzzy_targets.json',
    'artifacts/api/schemas.json',
    'artifacts/api/aliases.json',
]

all_ok = True
for p in artifacts:
    exists = Path(p).exists()
    size = f"{Path(p).stat().st_size/1024/1024:.1f} MB" if exists else "MISSING"
    print(f"{'OK' if exists else 'MISSING'} {p} ({size})")
    if not exists:
        all_ok = False

if not all_ok:
    print("\nRebuilding missing artifacts...")

In [ ]:
# ── Cell 4: (Chỉ chạy nếu thiếu artifacts) Rebuild doc index ────────────────
# Bỏ qua nếu Cell 3 báo OK hết
# !python rag/scripts/02_build_doc_index.py --force
# !python rag/scripts/03_build_api_index.py --force
print("Skip — artifacts already in repo")

In [ ]:
# ── Cell 5: Smoke test retriever (optional, ~2 phút) ────────────────────────
!python _test_retriever.py

In [ ]:
# ── Cell 6: Run eval trên 100 câu ───────────────────────────────────────────
# Lần đầu Qwen3-4B sẽ tải ~4GB từ HuggingFace (~5 phút trên Colab)
!python rag/scripts/06_eval.py

In [ ]:
# ── Cell 7: Xem metrics ─────────────────────────────────────────────────────
import json

with open('outputs/eval/metrics.json') as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))
print()
print(f"API JSON valid:    {metrics['api_json_valid_rate']:.1%}  ({int(metrics['api_json_valid_rate']*metrics['api_count'])}/{metrics['api_count']})")
print(f"Doc format valid:  {metrics['doc_format_valid_rate']:.1%}  ({int(metrics['doc_format_valid_rate']*metrics['doc_count'])}/{metrics['doc_count']})")
print(f"Avg time:          {metrics['avg_time_response']:.2f}s  (target <15s)")
print(f"Total wall time:   {metrics['total_wall_time']:.1f}s")

In [ ]:
# ── Cell 8: Xem 5 predictions đầu để debug ──────────────────────────────────
import json

with open('outputs/eval/predictions.jsonl') as f:
    preds = [json.loads(l) for l in f]

for p in preds[:5]:
    print(f"ID {p['id']}: {p['function_code']}")
    print(f"  result: {p['function_result'][:150]}")
    print()

In [ ]:
# ── Cell 9: (Optional) Run inference trên test set 617 câu ──────────────────
# !python rag/scripts/07_run_inference.py
print("Run 07_run_inference.py to generate submission file")